# A.2: Masked and unmasked loss

Compute the loss on response tokens only and on all tokens, for the base and the instruction-tuned model.

Earlier imports and definitions this listing relies on are included below so the notebook runs on its own.


In [ ]:
# Imports and definitions from earlier listings, included so this notebook runs alone.

# From A.1 Building labels with a loss mask
import torch

import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M-Instruct")

IGNORE = -100

def build_example(messages, tok):
    """Token IDs and labels for a conversation; only assistant turns (and their
    end-of-turn token) are trained, everything else gets the label IGNORE."""
    ids, labels = [], []
    for k in range(len(messages)):
        # Render the conversation up to and including turn k, and keep the new tokens.
        text = tok.apply_chat_template(messages[:k + 1], tokenize=False)
        new = tok(text, add_special_tokens=False).input_ids[len(ids):]
        if messages[k]["role"] == "assistant":
            header = tok.apply_chat_template(messages[:k], tokenize=False,
                                             add_generation_prompt=True)
            n_header = len(tok(header, add_special_tokens=False).input_ids) - len(ids)
            # The assistant header ("<|im_start|>assistant\n") is prompt, the rest is trained,
            # except the final newline after <|im_end|>, which the template adds between turns.
            body = new[n_header:]
            lab = [IGNORE] * n_header + body[:-1] + [IGNORE]
        else:
            lab = [IGNORE] * len(new)
        ids += new
        labels += lab
    return ids, labels

conversation = [
    {"role": "system", "content": "You are a concise assistant."},
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "Paris."},
    {"role": "user", "content": "And of Japan?"},
    {"role": "assistant", "content": "Tokyo."},
]

ids, labels = build_example(conversation, tok)

assert tok.decode(ids) == tok.apply_chat_template(conversation, tokenize=False)

trained = [i for i, l in zip(ids, labels) if l != IGNORE]

print("tokens:", len(ids), "| trained:", len(trained))

print("trained tokens:", [tok.decode([i]) for i in trained])


In [ ]:
def token_losses(model, ids):
    """Cross-entropy of each token given the ones before it (position t predicts t+1)."""
    x = torch.tensor([ids])
    with torch.no_grad():
        logits = model(x).logits[0, :-1]
    return F.cross_entropy(logits, x[0, 1:], reduction="none")

mask = torch.tensor([l != IGNORE for l in labels[1:]])
for name in ["HuggingFaceTB/SmolLM2-135M", "HuggingFaceTB/SmolLM2-135M-Instruct"]:
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.float32).eval()
    losses = token_losses(model, ids)
    print(f"{name.split('/')[1]:24s} response-only {losses[mask].mean():.3f}"
          f"  all tokens {losses.mean():.3f}  prompt-only {losses[~mask].mean():.3f}")


**Expected output**

Output:

```text
SmolLM2-135M             response-only 5.764  all tokens 5.784  prompt-only 5.787
SmolLM2-135M-Instruct    response-only 1.591  all tokens 1.694  prompt-only 1.711
```


In [ ]:
x, y = torch.tensor([ids]), torch.tensor([labels])
with torch.no_grad():
    hf_loss = model(input_ids=x, labels=y).loss
print(f"Hugging Face loss with IGNORE labels: {hf_loss:.3f}")


**Expected output**

Output:

```text
Hugging Face loss with IGNORE labels: 1.591
```
